# Token Embeddings

This notebook builds the first neural-network component of our Mini GPT.

So far, the tokenizer converts text into discrete token IDs. The model cannot directly reason over these integer IDs, so we map each token ID to a learnable continuous vector.

The pipeline becomes:

```text
Text
  ↓
Byte-Level BPE Tokenizer
  ↓
Token IDs
  ↓
Token Embeddings
  ↓
Transformer
```

The goal of this notebook is to understand what an embedding layer does, how it works internally, and how tokenized TinyStories text becomes the input representation for the Transformer.

---

## 1. Why Do We Need Embeddings?

Our tokenizer represents text as integer IDs.

For example:

```text
"Once upon a time"
        ↓
[430, 437, 259, 398]
```

These integers are identifiers, not meaningful numerical features.

Token `430` is not twice as meaningful as token `215`, and the numerical distance between token IDs has no semantic meaning.

Instead, the model maintains a learnable embedding vector for every token.

```text
Token ID
   ↓
Embedding lookup
   ↓
Continuous vector
```

If our vocabulary contains `8192` tokens and the model uses an embedding dimension of `256`, the embedding matrix has shape:

```text
[8192, 256]
```

Each row corresponds to one token.

---

## 2. Creating an Embedding Layer

PyTorch provides `nn.Embedding` for this purpose.

It can be thought of as a learnable lookup table:

```text
Embedding Matrix

Token 0   → vector
Token 1   → vector
Token 2   → vector
...
Token 430 → vector
...
Token 8191 → vector
```

Let's create an embedding layer for our Mini GPT configuration.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

VOCAB_SIZE = 8192
D_MODEL = 256

embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=D_MODEL
)

print("Embedding matrix shape:", embedding.weight.shape)

Embedding matrix shape: torch.Size([8192, 256])


## 3. What Does `nn.Embedding` Actually Do?

At its core, an embedding lookup selects rows from the embedding matrix.

For example, if the input token ID is:

```text
430
```

PyTorch retrieves row `430` from the embedding matrix.

Conceptually:

```python
vector = embedding.weight[430]
```

There is no mathematical transformation of the number `430` itself.

The token ID simply tells the model which row to retrieve.

This is why embeddings are different from passing token IDs directly through a numerical layer.

In [2]:
token_id = 430

vector = embedding.weight[token_id]

print("Token ID:", token_id)
print("Vector shape:", vector.shape)
print("First 10 values:", vector[:10])

Token ID: 430
Vector shape: torch.Size([256])
First 10 values: tensor([ 0.0391,  0.7450,  0.7679,  0.1398,  1.0090, -1.2032,  1.3852,  0.3487,
        -0.9193,  0.6670], grad_fn=<SliceBackward0>)


## 4. Embedding a Sequence of Tokens

A sequence of token IDs might look like:

```text
[430, 437, 259, 398]
```

Passing the entire sequence through the embedding layer performs four lookups:

```text
430 → embedding vector
437 → embedding vector
259 → embedding vector
398 → embedding vector
```

If `D_MODEL = 256`, the resulting tensor has shape:

```text
[4, 256]
```

The first dimension represents the sequence length.

The second dimension represents the embedding size.

In [3]:
token_ids = torch.tensor([430, 437, 259, 398])

embedded = embedding(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Embeddings shape:", embedded.shape)

Token IDs shape: torch.Size([4])
Embeddings shape: torch.Size([4, 256])


## 5. Batch and Sequence Dimensions

During training, we process multiple sequences at the same time.

The standard input shape is:

```text
[B, T]
```

where:

- `B` = batch size
- `T` = sequence length

The embedding layer adds the model dimension:

```text
[B, T]
   ↓
[B, T, D]
```

where:

- `D` = embedding dimension / model dimension

For example:

```text
Batch size     = 2
Sequence length = 8
Model dimension = 256

Input:
[2, 8]

After embedding:
[2, 8, 256]
```

In [4]:
B = 2
T = 8

token_ids = torch.randint(
    0,
    VOCAB_SIZE,
    (B, T)
)

embedded = embedding(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Embeddings shape:", embedded.shape)

Token IDs shape: torch.Size([2, 8])
Embeddings shape: torch.Size([2, 8, 256])


## 6. Connecting the TinyStories Tokenizer

Now we connect the tokenizer built in the previous phase.

Our complete pipeline is:

```text
TinyStories text
      ↓
Byte-Level BPE tokenizer
      ↓
Token IDs
      ↓
Embedding lookup
      ↓
Embedding vectors
```

The tokenizer and embedding layer have different responsibilities:

**Tokenizer**

Converts raw text into discrete token IDs.

**Embedding layer**

Converts those discrete token IDs into continuous learnable vectors.

This separation is important:

```text
Tokenizer:
"text" → [token IDs]

Model:
[token IDs] → [vectors]
```

In [7]:
import sys
sys.path.append("..")
from tokenizer.bpe import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
tokenizer.load("../artifacts/tinystories_bpe_8192.json")

text = "Once upon a time, there was a little dog."

tokens = tokenizer.encode(text)

print("Text:", text)
print("Token IDs:", tokens)
print("Number of tokens:", len(tokens))

Text: Once upon a time, there was a little dog.
Token IDs: [430, 437, 259, 398, 44, 400, 283, 259, 389, 464, 46]
Number of tokens: 11


## 7. From TinyStories Text to Embeddings

The tokenizer returns a one-dimensional list of token IDs.

For model training, we normally add a batch dimension.

```text
Token IDs

[T]

   ↓

Add batch dimension

[B, T]

   ↓

Embedding lookup

[B, T, D]
```

Let's run the complete pipeline.

In [8]:
token_ids = torch.tensor(tokens).unsqueeze(0)

embedded = embedding(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Embeddings shape:", embedded.shape)

Token IDs shape: torch.Size([1, 11])
Embeddings shape: torch.Size([1, 11, 256])


## 8. Inspecting the Embeddings

Each token now has a `D_MODEL`-dimensional representation.

For example, if the first token is `430`:

```text
Token ID: 430

        ↓

[0.12, -0.31, 0.07, ..., 0.44]
```

The individual values do not have a predefined meaning.

They are learned during training.

At initialization, the embedding vectors are essentially random. As the model trains, gradients update these vectors so that they become useful representations for the language modeling task.

In [9]:
first_token_id = token_ids[0, 0]

first_embedding = embedded[0, 0]

print("First token ID:", first_token_id.item())
print("Embedding shape:", first_embedding.shape)
print("First 10 values:", first_embedding[:10])

First token ID: 430
Embedding shape: torch.Size([256])
First 10 values: tensor([ 0.0391,  0.7450,  0.7679,  0.1398,  1.0090, -1.2032,  1.3852,  0.3487,
        -0.9193,  0.6670], grad_fn=<SliceBackward0>)


## 9. Embeddings Are Learnable Parameters

The embedding matrix is part of the model's parameters.

We can verify this directly:

```text
Embedding matrix
[8192, 256]

        ↓

Learnable model parameter
```

During training:

```text
Token IDs
   ↓
Embeddings
   ↓
Transformer
   ↓
Predictions
   ↓
Loss
   ↓
Gradients
   ↓
Update embedding weights
```

Therefore, embeddings are not fixed representations.

The model learns them from the training data.

In [10]:
print("Requires gradient:", embedding.weight.requires_grad)
print("Number of embedding parameters:", embedding.weight.numel())

Requires gradient: True
Number of embedding parameters: 2097152


## 10. Why Token IDs Cannot Simply Be Used Directly

Consider these token IDs:

```text
[10, 11, 12]
```

It would be incorrect to interpret them as numerical features where:

```text
12 > 11 > 10
```

The IDs are arbitrary assignments made by the tokenizer.

For example, token `500` might represent a common word fragment while token `501` might represent a completely unrelated byte sequence.

The embedding layer removes this arbitrary numerical interpretation by mapping each ID to its own learned vector.

The model therefore works with continuous representations rather than treating token IDs as meaningful scalar values.

## 11. Embedding Matrix vs. Tokenizer Vocabulary

It is useful to distinguish two different concepts.

### Tokenizer vocabulary

The tokenizer determines which token IDs exist.

For our tokenizer:

```text
Vocabulary size = 8192
```

### Embedding matrix

The model creates one learnable vector for each token ID.

```text
Embedding shape = [8192, 256]
```

The tokenizer defines the discrete vocabulary.

The model learns the numerical representation of that vocabulary.

```text
Tokenizer
    ↓
token ID 430
    ↓
Model embedding table
    ↓
256-dimensional vector
```

## 12. Shape Summary

For our Mini GPT:

```text
Vocabulary size = 8192
Model dimension = 256
```

The main transformations are:

```text
Text
 ↓
Tokenizer
 ↓
[T]
 ↓
Add batch dimension
 ↓
[B, T]
 ↓
Embedding
 ↓
[B, T, 256]
```

This `[B, T, D]` representation is what will eventually enter the Transformer blocks.

## 13. Key Takeaways

- Token IDs are discrete identifiers, not meaningful numerical features.
- An embedding layer maps every token ID to a learnable vector.
- The embedding matrix has shape `[vocab_size, d_model]`.
- Each row of the matrix represents one token.
- A sequence with shape `[T]` becomes `[T, d_model]`.
- A batch with shape `[B, T]` becomes `[B, T, d_model]`.
- Our TinyStories tokenizer provides the token IDs.
- The embedding layer provides the first learnable representation used by Mini GPT.
- Embedding vectors are updated through backpropagation during training.
- The next step is to add positional information so the model can distinguish token order.

The pipeline is now:

```text
Text
 ↓
Byte-Level BPE Tokenizer
 ↓
Token IDs
 ↓
Token Embeddings
 ↓
Positional Information
 ↓
Transformer
```